In [ ]:
# Prepare 4 dataframes: tmax_train_df, tmax_test_df, tmin_train_df, tmin_test_df
# FIX:include kriged monthly rasters under predictor set

import os
import gc
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import pyarrow as pa
import pyarrow.parquet as pq
from scipy.spatial import cKDTree
from tqdm import tqdm

start_year, start_month = 2001, 1
end_year, end_month = 2005, 12

min_valid_days = 20

monthly_raster_temp = r"..\Data_Temp\Processed\Monthly_downscaling\Input_rasters"
monthly_raster_prec = r"..\Data\Processed\Regression_model\Monthly_Rasters"

static_paths = {
    "elev_mean":       r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_mean.tif",
    "elev_std":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_std.tif",
    "elev_max":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif",
    "elev_min":        r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_min.tif",
    "slope_mean":      r"..\Data\Datasets\DEM_1km\DEM_1km_slope_mean.tif",
    "aspect_mean":     r"..\Data\Datasets\DEM_1km\DEM_1km_aspect_mean.tif",
    "elev_smoothed":   r"..\Data\Processed\Fetch\Preprocessing\dem_smoothed.tif",
    "fetch_1000m_dist":r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_1000m.tif",
    "fetch_1000m_dir": r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_1000m.tif",
    "fetch_500m_dist": r"..\Data\Processed\Fetch\Preprocessing\fetch_dist_500m.tif",
    "fetch_500m_dir":  r"..\Data\Processed\Fetch\Preprocessing\fetch_dir_500m.tif",
    "local_relief":    r"..\Data\Processed\Fetch\Preprocessing\local_relief.tif",
}

# In-situ
insitu_tmax_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_maxtemp_1991_2020.csv")
insitu_tmin_raw = pd.read_csv(r"..\Data\Processed\Insitu\insitu_mintemp_1991_2020.csv")
tmax_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmax.csv"
tmax_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmax.csv"

tmin_train_df = fr"..\Data_Temp\Processed\Split_stations\train_station_tmin.csv"
tmin_test_df = fr"..\Data_Temp\Processed\Split_stations\test_station_tmin.csv"

# Output directories
out_base = r"..\Data_Temp\Processed\Monthly_downscaling"

tmax_monthly_train_dir  = os.path.join(out_base, "tmax_monthly_dynamic_train_df")
tmin_monthly_train_dir  = os.path.join(out_base, "tmin_monthly_dynamic_train_df")

tmax_monthly_test_dir  = os.path.join(out_base, "tmax_monthly_dynamic_test_df")
tmin_monthly_test_dir  = os.path.join(out_base, "tmin_monthly_dynamic_test_df")

tmax_parquet_train_dir = os.path.join(out_base, "tmax_yearly_parquet_train")
tmin_parquet_train_dir = os.path.join(out_base, "tmin_yearly_parquet_train")

tmax_parquet_test_dir = os.path.join(out_base, "tmax_yearly_parquet_test")
tmin_parquet_test_dir = os.path.join(out_base, "tmin_yearly_parquet_test")

static_dir = os.path.join(out_base, "static_df")

for d in [tmax_monthly_train_dir, tmin_monthly_train_dir, tmax_monthly_test_dir, 
            tmin_monthly_test_dir, tmax_parquet_train_dir, tmin_parquet_train_dir,
            tmax_parquet_test_dir, tmin_parquet_test_dir, static_dir]:
    os.makedirs(d, exist_ok=True)

template = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif", masked = True)

In [2]:
# Helper functions
 
def month_cyclical(month: int):
    """Return (sin, cos) encoding for a calendar month (1–12)."""
    angle = 2 * np.pi * month / 12
    return np.sin(angle), np.cos(angle)
 
def mlabel(year: int, month: int) -> str:
    return f"{year}{month:02d}"

# Monthly raster path dictionary; divide that are in temp folder vs in prec folder

def build_monthly_paths_temp(year: int, month: int, t_variable: str) -> dict:
    """
    returns a dictionary of {feature_name: filepath} for all monthly rasters
    within the temperature folder (gridded temperature products)
    for the given year-month-variable (variable: tmax or tmin)
    """
    L = mlabel(year, month)
    R = monthly_raster_temp
    return {
        #Temperature average variables
        f"chirts_{t_variable}_average": f"{R}\\chirts\\chirts_{t_variable}_average_{L}.tif",
        f"era5l_{t_variable}_average": f"{R}\\era5l\\era5l_{t_variable}_average_{L}.tif",
        f"har_{t_variable}_average": f"{R}\\har\\har_{t_variable}_average_{L}.tif",

        #Temperature secondary variables from CHIRTS 
        f"chirts_{t_variable}_max": f"{R}\\chirts\\chirts_{t_variable}_max_{L}.tif",
        f"chirts_{t_variable}_min": f"{R}\\chirts\\chirts_{t_variable}_min_{L}.tif",
        f"chirts_{t_variable}_std": f"{R}\\chirts\\chirts_{t_variable}_std_{L}.tif",
        f"chirts_dtr_std": f"{R}\\chirts\\chirts_dtr_std_{L}.tif", 

        #Insitu 
        f"ik_{t_variable}_average": f"..\\Data_Temp\\Datasets\\Kriging_monthly\\{t_variable}\\{t_variable}_monthly_{L}.tif"     
    }

def build_monthly_paths_prec(year: int, month: int) -> dict:
    """
    returns a dictionary of {feature_name: filepath} for all monthly rasters
    within the precipitation folder (secondary meteorological variables and evi)
    for the given year-month
    """
    L = mlabel(year, month)
    R = monthly_raster_prec
    
    return {
        #GLDAS weather variables
        'gldas_air_temp':   f"{R}\\gldas_air_temp\\gldas_air_temp_monthly_mean_{L}.tif",
        'gldas_lwdown': f"{R}\\gldas_lwdown\\gldas_lwdown_monthly_sum_{L}.tif",
        'gldas_specific_humidity': f"{R}\\gldas_specific_humidity\\gldas_specific_humidity_monthly_mean_{L}.tif",
        'gldas_surface_pressure': f"{R}\\gldas_surface_pressure\\gldas_surface_pressure_monthly_mean_{L}.tif",
        'gldas_swdown': f"{R}\\gldas_swdown\\gldas_swdown_monthly_sum_{L}.tif",

        # MODIS EVI
        "evi_monthly_mean":         f"{R}\\modis_evi\\evi_monthly_mean_{L}.tif",
        "evi_missing_fraction":     f"{R}\\modis_evi\\evi_missing_fraction_{L}.tif",

        # MODIS LST
        "lst_d_monthly_mean":       f"{R}\\modis_lst\\lst_d_monthly_mean_{L}.tif",
        "lst_d_missing_fraction":   f"{R}\\modis_lst\\lst_d_missing_fraction_{L}.tif",
        "lst_n_monthly_mean":       f"{R}\\modis_lst\\lst_n_monthly_mean_{L}.tif",
        "lst_n_missing_fraction":   f"{R}\\modis_lst\\lst_n_missing_fraction_{L}.tif",
        "lst_diff_monthly_mean":    f"{R}\\modis_lst\\lst_diff_monthly_mean_{L}.tif",
        "lst_diff_missing_fraction":f"{R}\\modis_lst\\lst_diff_missing_fraction_{L}.tif"
    }

In [3]:
# 1 Build a static csv
static_csv = os.path.join(static_dir, "static_df.csv")
 
if os.path.exists(static_csv):
    print("Loading cached static df...")
    static_df = pd.read_csv(static_csv)
else:
    print("Building static df from rasters...")
    aligned = []
    for name, path in static_paths.items():
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da)
    static_xr = xr.merge(aligned, compat="override")
    static_df  = static_xr.to_dataframe().reset_index()
    static_df.to_csv(static_csv, index=False)
    del aligned, static_xr
    gc.collect()
 
static_df[["x", "y"]] = static_df[["x", "y"]].round(5)
print(f"Static df: {len(static_df):,} pixels")

Loading cached static df...
Static df: 484,619 pixels


In [4]:
# 2 Aggregate insitu monthly with completeness filter

print("==========Aggregating insitu tmax to monthly totals==============")
insitu_tmax_raw["date"] = pd.to_datetime(insitu_tmax_raw[['year', 'month', 'day']])
insitu_tmax_raw = insitu_tmax_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'max_temp': 'IS_tmax'})
insitu_tmax_raw = insitu_tmax_raw[insitu_tmax_raw["IS_tmax"] > -9999]

insitu_tmax_monthly = (
    insitu_tmax_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        IS_tmax_average = ('IS_tmax', 'mean'),
        valid_days = ('IS_tmax', 'count')
    )
    .reset_index()
)

n_before_tmax = len(insitu_tmax_monthly)
insitu_tmax_monthly = insitu_tmax_monthly[insitu_tmax_monthly["valid_days"] >= min_valid_days].copy()
n_after_tmax = len(insitu_tmax_monthly)
print(f"    Station-months before completeness filter: {n_before_tmax:,}")
print(f"    Station-months after (>= {min_valid_days} valid days): {n_after_tmax:,}")                                                       
print(f"    Excluded: {n_before_tmax - n_after_tmax:,} ({100*(n_before_tmax-n_after_tmax)/n_before_tmax:.1f}%)")

# Filter to analysis period
insitu_tmax_monthly = insitu_tmax_monthly[
    (insitu_tmax_monthly["year"]    >=  start_year) &
    (insitu_tmax_monthly["year"]    <=  end_year)
].copy()

insitu_tmax_monthly[["x", "y"]] = insitu_tmax_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmax = set(pd.read_csv(tmax_train_df)["index_no"])
test_ids_tmax = set(pd.read_csv(tmax_test_df)["index_no"])

insitu_tmax_monthly_train = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(train_ids_tmax)].copy()
insitu_tmax_monthly_test = insitu_tmax_monthly[insitu_tmax_monthly["index_no"].isin(test_ids_tmax)].copy()

print(f"\n  Train station-months: {len(insitu_tmax_monthly_train):,}")
print(f"  Test  station-months: {len(insitu_tmax_monthly_test):,}")

print("\n==========Aggregating insitu tmin to monthly totals==============")
insitu_tmin_raw["date"] = pd.to_datetime(insitu_tmin_raw[['year', 'month', 'day']])
insitu_tmin_raw = insitu_tmin_raw.rename(columns = {'lat': 'y', 'long': 'x',
                                                            'min_temp': 'IS_tmin'})
insitu_tmin_raw = insitu_tmin_raw[insitu_tmin_raw["IS_tmin"] >= -9999]

insitu_tmin_monthly = (
    insitu_tmin_raw
    .groupby(['index_no', 'y', "x", 'year', 'month'])
    .agg(
        IS_tmin_average = ('IS_tmin', 'mean'),
        valid_days = ('IS_tmin', 'count')
    )
    .reset_index()
)

n_before_tmin = len(insitu_tmin_monthly)
insitu_tmin_monthly = insitu_tmin_monthly[insitu_tmin_monthly["valid_days"] >= min_valid_days].copy()
n_after_tmin = len(insitu_tmin_monthly)
print(f"    Station-months before completeness filter: {n_before_tmin:,}")
print(f"    Station-months after (>= {min_valid_days} valid days): {n_after_tmin:,}")
print(f"    Excluded: {n_before_tmin - n_after_tmin:,} ({100*(n_before_tmin-n_after_tmin)/n_before_tmin:.1f}%)")

# Filter to analysis period
insitu_tmin_monthly = insitu_tmin_monthly[
    (insitu_tmin_monthly["year"]    >=  start_year) &
    (insitu_tmin_monthly["year"]    <=  end_year)
].copy()

insitu_tmin_monthly[["x", "y"]] = insitu_tmin_monthly[["x", "y"]].round(5)

# Train/test split
train_ids_tmin = set(pd.read_csv(tmin_train_df)["index_no"])
test_ids_tmin = set(pd.read_csv(tmin_test_df)["index_no"])

insitu_tmin_monthly_train = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(train_ids_tmin)].copy()
insitu_tmin_monthly_test = insitu_tmin_monthly[insitu_tmin_monthly["index_no"].isin(test_ids_tmin)].copy()

print(f"\n  Train station-months: {len(insitu_tmin_monthly_train):,}")
print(f"  Test  station-months: {len(insitu_tmin_monthly_test):,}")

==========Aggregating insitu tmax to monthly totals==============
    Station-months before completeness filter: 20,586
    Station-months after (>= 20 valid days): 20,429
    Excluded: 157 (0.8%)

  Train station-months: 2,073
  Test  station-months: 1,322

==========Aggregating insitu tmin to monthly totals==============
    Station-months before completeness filter: 20,379
    Station-months after (>= 20 valid days): 20,213
    Excluded: 166 (0.8%)

  Train station-months: 1,993
  Test  station-months: 1,318


In [5]:
# 3 Monthly loop to create aggregated file

month_range = pd.period_range(
    start = f"{start_year}-{start_month:02d}",
    end = f"{end_year}-{end_month:02d}",
    freq = "M"
)

print(f"\n ============ Processing {len(month_range)} months for Tmax =========")

for period in tqdm(month_range, desc="Monthly feature extraction"):
    year = period.year
    month = period.month
    label = mlabel(year, month)

    tmax_train_out = os.path.join(tmax_monthly_train_dir, f"tmax_monthly_{label}.csv")
    tmax_test_out = os.path.join(tmax_monthly_test_dir, f"tmax_monthly_{label}.csv")

    if os.path.exists(tmax_train_out) and os.path.exists(tmax_test_out):
        continue
    
    # A Load monthly rasters and flatten to pixel df
    paths_temp = build_monthly_paths_temp(year, month, "tmax")
    paths_prec = build_monthly_paths_prec(year, month)
    paths = paths_temp | paths_prec
    
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked = True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da.load())
        da.close()

    if missing_vars:
        tqdm.write(f"   [{label}] missing rasters:  {missing_vars}")

    if not aligned:
        tqdm.write(f"   [{label}]  no rasters found - skipping")
        continue

    # B Merge all dynamic rasters to a dataframe
    dynamic_xr = xr.merge(aligned, compat = 'override')
    dynamic_df = dynamic_xr.to_dataframe().reset_index()
    dynamic_df[['x', 'y']] = dynamic_df[['x', 'y']].round(5)

    for name in missing_vars:
        dynamic_df[name] = np.nan

    del aligned, dynamic_xr
    gc.collect()

    # C Merge with static features
    monthly_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = 'inner')
    sin_m, cos_m = month_cyclical(month)
    monthly_grid["month_sin"] = sin_m
    monthly_grid["month_cos"] = cos_m
    monthly_grid['year'] = year
    monthly_grid['month'] = month

    del dynamic_df
    gc.collect()

    # D cKDTree match to station locations
    # Threshold 0.006° ≈ ~600m at Nepal latitudes — one pixel tolerance. 
    insitu_train_month = insitu_tmax_monthly_train[
        (insitu_tmax_monthly_train['year'] == year) &
        (insitu_tmax_monthly_train['month'] == month) 
    ].copy()

    insitu_test_month = insitu_tmax_monthly_test[
        (insitu_tmax_monthly_test['year'] == year) &
        (insitu_tmax_monthly_test['month'] == month) 
    ].copy()

    if insitu_train_month.empty and insitu_test_month.empty:
        del monthly_grid
        gc.collect()
        continue
    
    tree = cKDTree(monthly_grid[['x', 'y']].values)

    if not insitu_train_month.empty:
        dist_tr, idx_tr = tree.query(insitu_train_month[['x', 'y']].values, k=1)
        mask_tr = dist_tr <= 0.006

        if mask_tr.any():
            station_rows = insitu_train_month.iloc[mask_tr].reset_index(drop=True)
            grid_rows = monthly_grid.iloc[idx_tr[mask_tr]].reset_index(drop=True)

            final_train = pd.concat([
                station_rows[['index_no', 'y', 'x', 'year', 'month',
                                'IS_tmax_average', 'valid_days'
                ]],
                grid_rows.drop(columns= ['x', 'y', 'year', 'month'])
            ], axis=1)

            final_train.to_csv(tmax_train_out, index = False)
    
    if not insitu_test_month.empty:
        dist_te, idx_te = tree.query(insitu_test_month[['x', 'y']].values, k=1)
        mask_te = dist_te <= 0.006

        if mask_te.any():
            station_rows = insitu_test_month.iloc[mask_te].reset_index(drop=True)
            grid_rows = monthly_grid.iloc[idx_te[mask_te]].reset_index(drop=True)

            final_test = pd.concat([
                station_rows[['index_no', 'y', 'x', 'year', 'month',
                                'IS_tmax_average', 'valid_days']],
                grid_rows.drop(columns=  ['x', 'y', 'year', 'month'])
            ], axis = 1)
            final_test.to_csv(tmax_test_out, index = False)
    
    del monthly_grid, insitu_train_month, insitu_test_month
    gc.collect()


 ============ Processing 60 months for Tmax =========


Monthly feature extraction: 100%|██████████| 60/60 [01:24<00:00,  1.41s/it]


In [6]:
print(f"\n ============ Processing {len(month_range)} months for Tmin =========")

for period in tqdm(month_range, desc="Monthly feature extraction"):
    year = period.year
    month = period.month
    label = mlabel(year, month)

    tmin_train_out = os.path.join(tmin_monthly_train_dir, f"tmin_monthly_{label}.csv")
    tmin_test_out = os.path.join(tmin_monthly_test_dir, f"tmin_monthly_{label}.csv")

    if os.path.exists(tmin_train_out) and os.path.exists(tmin_test_out):
        continue
    
    # A Load monthly rasters and flatten to pixel df
    paths_temp = build_monthly_paths_temp(year, month, "tmin")
    paths_prec = build_monthly_paths_prec(year, month)
    paths = paths_temp|paths_prec
    aligned = []
    missing_vars = []

    for name, path in paths.items():
        if not os.path.exists(path):
            missing_vars.append(name)
            continue
        da = rxr.open_rasterio(path, masked = True).rio.write_crs("EPSG:4326")
        da = da.rio.reproject_match(template).squeeze("band", drop=True)
        da.name = name
        aligned.append(da.load())
        da.close()

    if missing_vars:
        tqdm.write(f"   [{label}] missing rasters:  {missing_vars}")

    if not aligned:
        tqdm.write(f"   [{label}]  no rasters found - skipping")
        continue

    # B Merge all dynamic rasters to a dataframe
    dynamic_xr = xr.merge(aligned, compat = 'override')
    dynamic_df = dynamic_xr.to_dataframe().reset_index()
    dynamic_df[['x', 'y']] = dynamic_df[['x', 'y']].round(5)

    for name in missing_vars:
        dynamic_df[name] = np.nan

    del aligned, dynamic_xr
    gc.collect()

    # C Merge with static features
    monthly_grid = dynamic_df.merge(static_df, on = ['x', 'y'], how = 'inner')
    sin_m, cos_m = month_cyclical(month)
    monthly_grid["month_sin"] = sin_m
    monthly_grid["month_cos"] = cos_m
    monthly_grid['year'] = year
    monthly_grid['month'] = month

    del dynamic_df
    gc.collect()

    # D cKDTree match to station locations
    # Threshold 0.006° ≈ ~600m at Nepal latitudes — one pixel tolerance. 
    insitu_train_month = insitu_tmin_monthly_train[
        (insitu_tmin_monthly_train['year'] == year) &
        (insitu_tmin_monthly_train['month'] == month) 
    ].copy()

    insitu_test_month = insitu_tmin_monthly_test[
        (insitu_tmin_monthly_test['year'] == year) &
        (insitu_tmin_monthly_test['month'] == month) 
    ].copy()

    if insitu_train_month.empty and insitu_test_month.empty:
        del monthly_grid
        gc.collect()
        continue
    
    tree = cKDTree(monthly_grid[['x', 'y']].values)

    if not insitu_train_month.empty:
        dist_tr, idx_tr = tree.query(insitu_train_month[['x', 'y']].values, k=1)
        mask_tr = dist_tr <= 0.006

        if mask_tr.any():
            station_rows = insitu_train_month.iloc[mask_tr].reset_index(drop=True)
            grid_rows = monthly_grid.iloc[idx_tr[mask_tr]].reset_index(drop=True)

            final_train = pd.concat([
                station_rows[['index_no', 'y', 'x', 'year', 'month',
                                'IS_tmin_average', 'valid_days'
                ]],
                grid_rows.drop(columns= ['x', 'y', 'year', 'month'])
            ], axis=1)

            final_train.to_csv(tmin_train_out, index = False)
    
    if not insitu_test_month.empty:
        dist_te, idx_te = tree.query(insitu_test_month[['x', 'y']].values, k=1)
        mask_te = dist_te <= 0.006

        if mask_te.any():
            station_rows = insitu_test_month.iloc[mask_te].reset_index(drop=True)
            grid_rows = monthly_grid.iloc[idx_te[mask_te]].reset_index(drop=True)

            final_test = pd.concat([
                station_rows[['index_no', 'y', 'x', 'year', 'month',
                                'IS_tmin_average', 'valid_days']],
                grid_rows.drop(columns=  ['x', 'y', 'year', 'month'])
            ], axis = 1)
            final_test.to_csv(tmin_test_out, index = False)
    
    del monthly_grid, insitu_train_month, insitu_test_month
    gc.collect()


 ============ Processing 60 months for Tmin =========


Monthly feature extraction: 100%|██████████| 60/60 [01:10<00:00,  1.17s/it]


In [ ]:
# 5 Aggregate monthly CSVs to yearly parquet --> final combined df

def aggregate_to_parquet(monthly_dir: str, parquet_dir: str, t_variable: str, split_label:str) -> str:
    """
    Read monthly CSVs year by year, write to parquet, then concatenate
    all years into a single final df. Returns path to final parquet.
    """
    parquet_paths = []

    for year in range(start_year, end_year + 1):
        yearly_path = os.path.join(parquet_dir, f"full_{year}.parquet")
        monthly_periods = pd.period_range(
            start = f"{year}-01", end = f"{year}-12", freq = "M"
        )
        expected = [
        os.path.join(monthly_dir, f"{t_variable}_monthly_{mlabel(p.year, p.month)}.csv")
        for p in monthly_periods
        ]
        found = [f for f in expected if os.path.exists(f)]

        if not found:
            print(f"  {t_variable} - [{split_label}]  {year} - no monthly files found, skipping")
            continue

        writer = None
        for fpath in found:
            chunk = pd.read_csv(fpath)
            table = pa.Table.from_pandas(chunk)
            if writer is None:
                writer = pq.ParquetWriter(yearly_path, table.schema)
            writer.write_table(table)
        
        if writer:
            writer.close()
            parquet_paths.append(yearly_path)
            print(f"    [{split_label}] {year} - {len(found)} months written to parquet")

    if not parquet_paths:
        print(f"    [{split_label}] No data - final df not written")
        return None
    
    final_df = pd.concat([pd.read_parquet(p) for p in parquet_paths], ignore_index = True)
    final_path = os.path.join(out_base, f"final_{t_variable}_{split_label}_df.parquet")
    final_df.to_parquet(final_path, index=False)

    print(f"\n  [{t_variable} - {split_label}] Final df: {len(final_df):,} rows × {len(final_df.columns)} cols")
    print(f"  Saved: {final_path}")
    return final_path

print("\n ================Aggregating to yearly parquet files================")

tmax_train_final_path = aggregate_to_parquet(tmax_monthly_train_dir, tmax_parquet_train_dir, "tmax", "train")
tmax_test_final_path = aggregate_to_parquet(tmax_monthly_test_dir, tmax_parquet_test_dir, "tmax", "test")
tmin_train_final_path = aggregate_to_parquet(tmin_monthly_train_dir, tmin_parquet_train_dir, "tmin", "train")
tmin_test_final_path = aggregate_to_parquet(tmin_monthly_test_dir, _parquet_test_dir, "tmin", "test")


 ================Aggregating to yearly parquet files================
    [train] 2001 - 12 months written to parquet
    [train] 2002 - 12 months written to parquet
    [train] 2003 - 12 months written to parquet
    [train] 2004 - 12 months written to parquet
    [train] 2005 - 12 months written to parquet

  [tmax - train] Final df: 2,073 rows × 43 cols
  Saved: ..\Data_Temp\Processed\Monthly_downscaling\final_tmax_train_df.parquet
    [test] 2001 - 12 months written to parquet
    [test] 2002 - 12 months written to parquet
    [test] 2003 - 12 months written to parquet
    [test] 2004 - 12 months written to parquet
    [test] 2005 - 12 months written to parquet

  [tmax - test] Final df: 1,322 rows × 43 cols
  Saved: ..\Data_Temp\Processed\Monthly_downscaling\final_tmax_test_df.parquet
    [train] 2001 - 12 months written to parquet
    [train] 2002 - 12 months written to parquet
    [train] 2003 - 12 months written to parquet
    [train] 2004 - 12 months written to parquet
    [tr

In [13]:
# ── 7. Clean up intermediate files ──────────────────────────────────────────────

import shutil

print("\n" + "=" * 60)
print("Cleaning up intermediate files and directories...")
print("=" * 60)

dirs_to_remove = [
    tmax_monthly_train_dir,
    tmin_monthly_train_dir,
    tmax_monthly_test_dir,
    tmin_monthly_test_dir,
    tmax_parquet_train_dir,
    tmin_parquet_train_dir,
    tmax_parquet_test_dir,
    tmin_parquet_test_dir,
    static_dir
]

for d in dirs_to_remove:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"Removed directory: {d}")

# # Also remove preview csvs
# for split in ['train', 'test']:
#     preview_csv = os.path.join(OUT_BASE, f"final_{split}_df_preview.csv")
#     if os.path.exists(preview_csv):
#         os.remove(preview_csv)
#         print(f"Removed preview CSV: {preview_csv}")

print("\nCleanup complete.")



Cleaning up intermediate files and directories...
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmax_monthly_dynamic_train_df
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmin_monthly_dynamic_train_df
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmax_monthly_dynamic_test_df
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmin_monthly_dynamic_test_df
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmax_yearly_parquet_train
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmin_yearly_parquet_train
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmax_yearly_parquet_test
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\tmin_yearly_parquet_test
Removed directory: ..\Data_Temp\Processed\Monthly_downscaling\static_df

Cleanup complete.
